# Hostile Testing Phase 2 - Config, Git, Admin, Development

## Purpose
Expand hostile testing coverage from 1.1% to 25%+ by testing:
- Config system functions (databases, projects, directories, clients, connections)
- Git utilities (branch analysis, operations)
- Admin functions (profile management)
- Development utilities (architecture analysis)

## Goal
Test ~180 more functions to reach 25% coverage (188/751 functions)

In [ ]:
import sys
import pandas as pd
import tempfile
from pathlib import Path
import shutil
import json

# Load test results from Phase 1
try:
    phase1_results = pd.read_csv('../hostile_testing_results.csv')
    print(f"Phase 1 completed: {len(phase1_results)} tests")
except FileNotFoundError:
    phase1_results = pd.DataFrame()
    print("Starting fresh")

# Test results tracking
test_results = {
    'function': [],
    'module': [],
    'test_category': [],
    'passed': [],
    'error_message': [],
    'severity': []
}

def record_test(function, module, test_category, passed, error_message="", severity="medium"):
    """Record a test result."""
    test_results['function'].append(function)
    test_results['module'].append(module)
    test_results['test_category'].append(test_category)
    test_results['passed'].append(passed)
    test_results['error_message'].append(error_message)
    test_results['severity'].append(severity)

def hostile_test(func, test_name, *args, **kwargs):
    """Execute a hostile test and record results."""
    try:
        result = func(*args, **kwargs)
        return (True, result, "")
    except Exception as e:
        return (False, None, str(e))

print("Test harness loaded for Phase 2")

## Section 1: Config System - Database Configuration

In [ ]:
from siege_utilities import (
    create_database_config, save_database_config, load_database_config,
    get_spark_database_options, list_database_configs
)

test_dir = Path(tempfile.mkdtemp())

try:
    # Hostile database config inputs
    hostile_db_configs = [
        # SQL Injection attempts in database names
        {"name": "'; DROP TABLE users; --", "type": "postgresql", "host": "localhost"},
        {"name": "<script>alert('xss')</script>", "type": "mysql", "host": "localhost"},
        # Path traversal in connection strings
        {"name": "test", "type": "sqlite", "database": "../../../etc/passwd"},
        # Very long strings
        {"name": "A" * 10000, "type": "postgresql", "host": "localhost"},
        # Null bytes
        {"name": "test\x00db", "type": "mysql", "host": "localhost\x00evil"},
        # Empty/None values
        {"name": "", "type": "postgresql", "host": ""},
        {"name": None, "type": None, "host": None},
        # Command injection in host
        {"name": "test", "type": "postgresql", "host": "localhost; rm -rf /"},
        # Unicode and special characters
        {"name": "数据库", "type": "postgresql", "host": "本地主机"},
        # URL-encoded attacks
        {"name": "test%00admin", "type": "mysql", "host": "localhost%2F.."},
    ]
    
    for i, config in enumerate(hostile_db_configs):
        # Test create_database_config
        success, result, error = hostile_test(
            create_database_config,
            f"hostile_db_config_{i}",
            **config
        )
        record_test(
            "create_database_config",
            "config.databases",
            "hostile_db_configs",
            success,
            error,
            severity="high" if "DROP TABLE" in str(config.get('name', '')) else "medium"
        )
        
        # If config was created, try to save it
        if success and result:
            save_path = test_dir / f"db_config_{i}.json"
            success2, result2, error2 = hostile_test(
                save_database_config,
                f"save_hostile_db_{i}",
                result,
                save_path
            )
            record_test(
                "save_database_config",
                "config.databases",
                "save_hostile_configs",
                success2,
                error2,
                severity="medium"
            )
    
    # Test loading non-existent configs
    success, result, error = hostile_test(
        load_database_config,
        "load_nonexistent",
        test_dir / "nonexistent.json"
    )
    record_test(
        "load_database_config",
        "config.databases",
        "nonexistent_file",
        success,
        error,
        severity="low"
    )
    
    # Test loading corrupt JSON
    corrupt_file = test_dir / "corrupt.json"
    corrupt_file.write_text("{invalid json}}}}")
    success, result, error = hostile_test(
        load_database_config,
        "load_corrupt",
        corrupt_file
    )
    record_test(
        "load_database_config",
        "config.databases",
        "corrupt_json",
        success,
        error,
        severity="medium"
    )
    
    print(f"Completed {len([r for r in test_results['module'] if r == 'config.databases'])} database config tests")
    
finally:
    shutil.rmtree(test_dir, ignore_errors=True)

## Section 2: Config System - Project Configuration

In [ ]:
from siege_utilities import (
    create_project_config, save_project_config, load_project_config,
    list_projects, update_project_config
)

test_dir = Path(tempfile.mkdtemp())

try:
    # Hostile project inputs
    hostile_projects = [
        {"name": "../../../etc/passwd", "path": "/tmp/test"},
        {"name": "test; rm -rf /", "path": "/tmp/test"},
        {"name": "A" * 1000, "path": "B" * 1000},
        {"name": "test\x00project", "path": "/tmp\x00/evil"},
        {"name": "", "path": ""},
        {"name": None, "path": None},
        {"name": "<script>alert('xss')</script>", "path": "/tmp/test"},
        {"name": "test", "path": "file:///etc/shadow"},
    ]
    
    for i, project in enumerate(hostile_projects):
        success, result, error = hostile_test(
            create_project_config,
            f"hostile_project_{i}",
            **project
        )
        record_test(
            "create_project_config",
            "config.projects",
            "hostile_projects",
            success,
            error,
            severity="high" if "../../../" in str(project.get('name', '')) else "medium"
        )
    
    print(f"Completed {len([r for r in test_results['module'] if r == 'config.projects'])} project config tests")
    
finally:
    shutil.rmtree(test_dir, ignore_errors=True)

## Section 3: Config System - Client Profiles

In [ ]:
from siege_utilities import (
    create_client_profile, save_client_profile, load_client_profile,
    list_client_profiles, validate_client_profile
)

test_dir = Path(tempfile.mkdtemp())

try:
    # Hostile client profile inputs
    hostile_clients = [
        {"name": "'; DROP TABLE clients; --", "email": "test@example.com"},
        {"name": "test", "email": "<script>alert('xss')</script>@evil.com"},
        {"name": "../../../root", "email": "test@example.com"},
        {"name": "A" * 10000, "email": "B" * 10000 + "@test.com"},
        {"name": "test\x00client", "email": "test\x00@evil.com"},
        {"name": "", "email": ""},
        {"name": None, "email": None},
        # Email injection attempts
        {"name": "test", "email": "test@example.com\nBCC: attacker@evil.com"},
        {"name": "test", "email": "test@example.com\r\nTo: victim@target.com"},
    ]
    
    for i, client in enumerate(hostile_clients):
        success, result, error = hostile_test(
            create_client_profile,
            f"hostile_client_{i}",
            **client
        )
        record_test(
            "create_client_profile",
            "config.clients",
            "hostile_clients",
            success,
            error,
            severity="critical" if "DROP TABLE" in str(client.get('name', '')) else "medium"
        )
        
        # Test validation if profile was created
        if success and result:
            success2, result2, error2 = hostile_test(
                validate_client_profile,
                f"validate_hostile_{i}",
                result
            )
            record_test(
                "validate_client_profile",
                "config.clients",
                "validate_hostile",
                success2,
                error2,
                severity="medium"
            )
    
    print(f"Completed {len([r for r in test_results['module'] if r == 'config.clients'])} client profile tests")
    
finally:
    shutil.rmtree(test_dir, ignore_errors=True)

## Section 4: Git Utilities - Branch Analysis

In [ ]:
from siege_utilities import analyze_branch_status, generate_branch_report

# Hostile repository paths
hostile_repo_paths = [
    None,  # Null
    "",  # Empty
    "../../../etc/passwd",  # Path traversal
    "/nonexistent/repo",  # Non-existent path
    "/dev/null",  # Special device
    "repo; rm -rf /",  # Command injection
    "A" * 10000,  # Very long path
    "repo\x00path",  # Null byte
    "~/../../../../../../etc/shadow",  # Complex traversal
    ".git",  # Git directory itself
]

for repo_path in hostile_repo_paths:
    # Test analyze_branch_status
    success, result, error = hostile_test(
        analyze_branch_status,
        "hostile_repo_path",
        repo_path
    )
    record_test(
        "analyze_branch_status",
        "git.branch_analyzer",
        "hostile_repo_paths",
        success,
        error,
        severity="high" if "../../../" in str(repo_path) else "medium"
    )
    
    # Test generate_branch_report
    success2, result2, error2 = hostile_test(
        generate_branch_report,
        "hostile_repo_report",
        repo_path
    )
    record_test(
        "generate_branch_report",
        "git.branch_analyzer",
        "hostile_repo_paths",
        success2,
        error2,
        severity="high" if "../../../" in str(repo_path) else "medium"
    )

print(f"Completed {len([r for r in test_results['module'] if 'git' in r])} git utility tests")

## Section 5: Admin Functions - Profile Management

In [ ]:
from siege_utilities import (
    get_default_profile_location, set_profile_location,
    get_profile_location, validate_profile_location
)

# Hostile profile locations
hostile_locations = [
    None,
    "",
    "../../../etc/passwd",
    "/dev/null",
    "location; rm -rf /",
    "A" * 10000,
    "path\x00injection",
    "~/.ssh/id_rsa",  # SSH key
    "/etc/shadow",  # Shadow file
    "file:///etc/passwd",  # File URL
]

for location in hostile_locations:
    # Test set_profile_location
    success, result, error = hostile_test(
        set_profile_location,
        "hostile_location",
        location
    )
    record_test(
        "set_profile_location",
        "admin.profile_manager",
        "hostile_locations",
        success,
        error,
        severity="critical" if "ssh" in str(location) or "shadow" in str(location) else "high"
    )
    
    # Test validate_profile_location
    success2, result2, error2 = hostile_test(
        validate_profile_location,
        "validate_hostile",
        location
    )
    record_test(
        "validate_profile_location",
        "admin.profile_manager",
        "hostile_locations",
        success2,
        error2,
        severity="high"
    )

# Test get_default_profile_location with no arguments
success, result, error = hostile_test(
    get_default_profile_location,
    "get_default"
)
record_test(
    "get_default_profile_location",
    "admin.profile_manager",
    "no_args",
    success,
    error,
    severity="low"
)

print(f"Completed {len([r for r in test_results['module'] if 'admin' in r])} admin function tests")

## Section 6: User Config System

In [ ]:
from siege_utilities import get_user_config, get_download_directory

# Test get_user_config with various inputs
hostile_config_files = [
    None,
    "",
    "../../../etc/passwd",
    "/nonexistent/config.json",
    "config\x00.json",
    "A" * 10000 + ".json",
]

for config_file in hostile_config_files:
    success, result, error = hostile_test(
        get_user_config,
        "hostile_config",
        config_file
    )
    record_test(
        "get_user_config",
        "config.user_config",
        "hostile_config_files",
        success,
        error,
        severity="high" if "../../../" in str(config_file) else "medium"
    )

# Test get_download_directory
hostile_client_codes = [
    None,
    "",
    "../../../etc",
    "client; rm -rf /",
    "A" * 10000,
    "client\x00code",
]

for code in hostile_client_codes:
    success, result, error = hostile_test(
        get_download_directory,
        "hostile_client_code",
        code
    )
    record_test(
        "get_download_directory",
        "config.user_config",
        "hostile_client_codes",
        success,
        error,
        severity="high" if "../../../" in str(code) else "medium"
    )

print(f"Completed user config tests")

## Section 7: Generate Phase 2 Results

In [ ]:
# Convert results to DataFrame
df_phase2 = pd.DataFrame(test_results)

# Combine with Phase 1 if available
if len(phase1_results) > 0:
    df_combined = pd.concat([phase1_results, df_phase2], ignore_index=True)
else:
    df_combined = df_phase2

# Summary statistics
print("\n" + "="*80)
print("PHASE 2 HOSTILE TESTING SUMMARY")
print("="*80)
print(f"\nPhase 2 tests run: {len(df_phase2)}")
print(f"Phase 2 passed: {df_phase2['passed'].sum()}")
print(f"Phase 2 failed: {(~df_phase2['passed']).sum()}")
print(f"Phase 2 pass rate: {df_phase2['passed'].sum() / len(df_phase2) * 100:.1f}%")

print("\n" + "="*80)
print("COMBINED RESULTS (Phase 1 + Phase 2)")
print("="*80)
print(f"\nTotal tests run: {len(df_combined)}")
print(f"Total passed: {df_combined['passed'].sum()}")
print(f"Total failed: {(~df_combined['passed']).sum()}")
print(f"Overall pass rate: {df_combined['passed'].sum() / len(df_combined) * 100:.1f}%")

# Functions tested
unique_functions = df_combined['function'].nunique()
print(f"\nUnique functions tested: {unique_functions}")
print(f"Coverage: {unique_functions}/751 = {unique_functions/751*100:.1f}%")

# Summary by module
print("\n" + "="*80)
print("RESULTS BY MODULE")
print("="*80)
module_summary = df_combined.groupby('module').agg({
    'passed': ['sum', 'count']
})
module_summary.columns = ['passed', 'total']
module_summary['pass_rate'] = (module_summary['passed'] / module_summary['total'] * 100).round(1)
print(module_summary)

# Critical failures
print("\n" + "="*80)
print("FAILURES BY SEVERITY")
print("="*80)
failures = df_combined[~df_combined['passed']]
if len(failures) > 0:
    severity_summary = failures.groupby('severity').size()
    print(severity_summary)
    
    print("\n" + "="*80)
    print("CRITICAL AND HIGH SEVERITY FAILURES")
    print("="*80)
    critical_high = failures[failures['severity'].isin(['critical', 'high'])]
    if len(critical_high) > 0:
        print(critical_high[['function', 'module', 'test_category', 'error_message', 'severity']].to_string())
    else:
        print("No critical or high severity failures")
else:
    print("No failures detected")

# Save combined results
results_file = Path("hostile_testing_phase2_results.csv")
df_combined.to_csv(results_file, index=False)
print(f"\nCombined results saved to: {results_file}")

# Display sample
print("\n" + "="*80)
print("PHASE 2 SAMPLE RESULTS")
print("="*80)
print(df_phase2.head(20))

## Section 8: Next Steps

Based on Phase 2 results:
1. Fix any critical or high severity failures
2. Continue to Phase 3: Test remaining modules (data, hygiene, development)
3. Reach 50% coverage goal
4. Document all findings